# Component 4 – Tea Maturity Classifier (Roboflow dataset)

**Function 1:** field photo → maturity (Young / Medium / Mature) + confidence.  
**Function 2 check:** adaptive spraying vs fixed-rate on the test set.

Uses the **Roboflow classification export** and keeps Roboflow's own train / valid / test split.

Before running: **Runtime → Change runtime type → T4 GPU**, and push your latest code to GitHub.

## 1. Drive and code

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA = '/content/drive/MyDrive/C4_data'

In [ ]:
# If the repo is private, paste a read-only GitHub token when asked (never save it in the notebook).
import getpass, os, subprocess
REPO = 'https://github.com/ShiharaXcor/AI-Driven-Drone-System-for-Precision-Spraying-in-Sri-Lankan-Tea-Plantations-J26-DS-346.git'
BRANCH = 'component4/maturity-spray-engine'
if not os.path.exists('/content/repo'):
    token = getpass.getpass('GitHub token (leave empty if repo is public): ')
    url = REPO.replace('https://', f'https://{token}@') if token else REPO
    subprocess.run(['git', 'clone', '-q', '--branch', BRANCH, url, '/content/repo'], check=True)
    del token, url
%cd /content/repo/component4_maturity_engine
!git pull -q
!git log --oneline -3
!pip -q install timm scikit-learn pyyaml

## 2. Load the Roboflow export

Set `ROBOFLOW` to the zip (or extracted folder) you uploaded to `MyDrive/C4_data`. Leave it empty to pick the zip automatically if there is only one.

In [ ]:
import glob
ROBOFLOW = ''   # e.g. f'{DATA}/tea-maturity.v1i.folder.zip'

if not ROBOFLOW:
    zips = [z for z in glob.glob(f'{DATA}/*.zip') if 'training_pack' not in z and 'tiles' not in z]
    print('Zips in C4_data:', [os.path.basename(z) for z in zips])
    assert len(zips) == 1, 'Set ROBOFLOW to the right file above'
    ROBOFLOW = zips[0]
print('Using', ROBOFLOW)

!rm -rf data/roboflow && mkdir -p data/roboflow
if ROBOFLOW.endswith('.zip'):
    !unzip -q -o "$ROBOFLOW" -d data/roboflow
else:
    !cp -r "$ROBOFLOW"/. data/roboflow/
!ls data/roboflow

## 3. Check the dataset

Class counts per split and a **leakage check**: the same source photo in train and test would make test accuracy look better than it really is.

In [ ]:
from src.classifier.roboflow import load_roboflow, describe_roboflow, leakage_report
split = load_roboflow('data/roboflow')
print(describe_roboflow(split))

In [ ]:
# Look at a few training images per class - make sure the labels look right
import random, matplotlib.pyplot as plt
from PIL import Image
from src.classifier.dataset import CLASSES
random.seed(0)
fig, axes = plt.subplots(3, 6, figsize=(15, 8))
for r, c in enumerate(CLASSES):
    imgs = [p for p, l, _ in split['train'] if CLASSES[l] == c]
    for k, ax in enumerate(axes[r]):
        ax.axis('off')
        if k < len(imgs):
            ax.imshow(Image.open(random.choice(imgs)).convert('RGB'))
            if k == 0: ax.set_title(c.upper(), loc='left')
plt.tight_layout(); plt.show()

## 4. Train EfficientNet-B0 (main model)

In [ ]:
!python -m src.classifier.train --roboflow data/roboflow --model efficientnet_b0 \
    --epochs 30 --batch 32 --patience 8 --out runs/effnet_b0

## 5. Train MobileNetV3 (comparison model)

In [ ]:
!python -m src.classifier.train --roboflow data/roboflow --model mobilenetv3_large_100 \
    --epochs 30 --batch 32 --patience 8 --out runs/mobilenetv3

## 6. Compare models

In [ ]:
import json, pandas as pd
from IPython.display import Image as Img, display
rows = []
for run in ['runs/effnet_b0', 'runs/mobilenetv3']:
    r = json.load(open(f'{run}/results.json'))
    rows.append({'model': r['model'], 'accuracy': r['accuracy'], 'macro_precision': r['macro_precision'],
                 'macro_recall': r['macro_recall'], 'macro_f1': r['macro_f1'],
                 'young_recall': r['per_class']['young']['recall'],
                 'latency_ms': r['latency_ms_per_tile'], 'test_images': r['test_tiles']})
    display(Img(f'{run}/confusion_matrix.png')); display(Img(f'{run}/training_curves.png'))
pd.DataFrame(rows)

## 7. Look at the mistakes

Predicts every test image with the best model and shows the ones it got wrong. Use this to decide whether the problem is the model or the labels.

In [ ]:
!python -m src.classifier.predict --model runs/effnet_b0/best.pt \
    --split runs/effnet_b0/splits.csv --out runs/effnet_b0/test_predictions.csv
pred = pd.read_csv('runs/effnet_b0/test_predictions.csv')
paths = pd.read_csv('runs/effnet_b0/splits.csv').query("split == 'test'").set_index('image')['path']
wrong = pred[pred.true_label != pred.predicted].sort_values('confidence', ascending=False)
print(f'{len(wrong)} of {len(pred)} test images wrong')
n = min(len(wrong), 12)
if n:
    fig, axes = plt.subplots((n + 3) // 4, 4, figsize=(14, 3.6 * ((n + 3) // 4)), squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax, (_, r) in zip(axes.flat, wrong.head(n).iterrows()):
        ax.imshow(Image.open(paths[r.image]).convert('RGB'))
        ax.set_title(f'true {r.true_label} / pred {r.predicted} ({r.confidence:.0%})', fontsize=9)
    plt.tight_layout(); plt.show()

## 8. Function 2 – adaptive spraying vs fixed rate

In [ ]:
!python -m src.prescription.evaluate --predictions runs/effnet_b0/test_predictions.csv \
    --out runs/effnet_b0/baseline_comparison
r = json.load(open('runs/effnet_b0/baseline_comparison/comparison.json'))
display(Img('runs/effnet_b0/baseline_comparison/comparison.png'))
pd.DataFrame({k: r[k] for k in ['fixed_rate_baseline', 'adaptive_engine']}).T

## 9. Save to Drive

Saved as `C4_data/runs_v2` so the first (self-labelled) results in `C4_data/runs` are kept for comparison.

In [ ]:
!mkdir -p $DATA/runs_v2 && cp -r runs/effnet_b0 runs/mobilenetv3 $DATA/runs_v2/
!ls $DATA/runs_v2/effnet_b0